# SwiGLU Activation
*The gated feedforward activation in LLaMA, PaLM, Gemma, and most modern LLMs.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_vec4_load.py`


## What Is SwiGLU?

**In plain English:** SwiGLU is a "gating" activation function. The feedforward layer produces two parallel outputs: a "gate" vector and an "up" vector. The gate is passed through SiLU (a smooth activation), then element-wise multiplied with the up vector. The gate acts like a learned dimmer switch — it controls how much of each feature passes through.

**Why it matters:** The original transformer used ReLU. SwiGLU (Noam Shazeer, 2020) consistently outperforms both GELU and ReLU at the same parameter count, with no theoretical justification — it just works.


In [ ]:
import math
print('ready')

## Building Blocks: SiLU and GLU

### SiLU (Sigmoid Linear Unit)

$$\text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

- When $x \gg 0$: $\sigma(x) \to 1$, so $\text{SiLU}(x) \approx x$ (identity — passes through)
- When $x \approx 0$: $\sigma(0) = 0.5$, so $\text{SiLU}(0) = 0$ (attenuated)
- When $x \ll 0$: $\sigma(x) \to 0$, so $\text{SiLU}(x) \approx 0$ (suppressed)

### GLU (Gated Linear Unit, Dauphin et al. 2017)

$$\text{GLU}(a, b) = a \otimes \sigma(b)$$

SwiGLU replaces $\sigma$ with SiLU:

$$\boxed{\text{SwiGLU}(x, y) = \text{SiLU}(x) \otimes y = \frac{x}{1+e^{-x}} \otimes y}$$


In [ ]:
def silu(x):
    return x / (1.0 + math.exp(-x))

def swiglu(gate, up):
    return silu(gate) * up

# Trace a single element
print("SiLU behavior:")
for x in [-3.0, -1.0, 0.0, 1.0, 3.0]:
    s = silu(x)
    print(f"  SiLU({x:+.1f}) = {s:+.4f}  "
          f"({'≈x' if abs(s-x)<0.1 else '≈0' if abs(s)<0.1 else ''})")

print()
# Element-wise example
gate = [2.0, -1.0, 0.5, 3.0]
up   = [1.5,  2.0, 0.8, 0.3]
print("SwiGLU element-wise:")
print(f"  gate:      {gate}")
print(f"  SiLU(gate): {[round(silu(g),3) for g in gate]}")
out = [swiglu(g, u) for g, u in zip(gate, up)]
print(f"  × up:      {[round(v,3) for v in out]}")


## The Formula

$$\boxed{\text{SwiGLU}(x, y) = \text{SiLU}(x) \cdot y = \frac{x}{1 + e^{-x}} \cdot y}$$

| Symbol | Meaning |
|--------|---------|
| $x$ | Gate vector (from one linear projection of the input) |
| $y$ | Up-projection vector (from another linear projection) |
| $\sigma(x) = \frac{1}{1+e^{-x}}$ | Sigmoid — squashes $x$ to $(0,1)$ |
| $\text{SiLU}(x) = x \cdot \sigma(x)$ | Smooth, differentiable, no "dead neuron" problem |
| $\otimes$ | Element-wise multiplication |

### 🗣️ Say It Out Loud
> *"SwiGLU of x and y equals x times sigma of x, times y — where sigma is the sigmoid function, one over one plus e to the minus x."*

**Tutor's gloss:** "Think of $x$ as the gate controlling a valve and $y$ as the flow through it. When the gate $x$ is large positive, $\sigma(x) \approx 1$, so the valve is fully open: $y$ passes through unchanged. When $x$ is near zero, the valve is half-open. When $x$ is large negative, it's nearly closed: $y$ is suppressed. The model learns to set the gate values to route information."


## SwiGLU in Context: The LLaMA FFN

A standard transformer FFN layer:
$$\text{FFN}(x) = W_\text{down} \cdot \text{ReLU}(W_\text{up} \cdot x)$$

LLaMA's SwiGLU FFN:
$$\text{FFN}_\text{SwiGLU}(x) = W_\text{down} \cdot \bigl(\text{SiLU}(W_\text{gate} \cdot x) \otimes W_\text{up} \cdot x\bigr)$$

Three weight matrices instead of two. To keep the same parameter count, LLaMA uses:

$$d_\text{hidden} = \left\lfloor \frac{8}{3} d_\text{model} \right\rfloor \approx 2.67 \times d_\text{model}$$

instead of the usual $4 \times d_\text{model}$.

| Variant | Activation | Weight matrices | Typical hidden dim |
|---------|-----------|-----------------|-------------------|
| Original Transformer | ReLU | 2 | $4d$ |
| GPT-2/3 | GELU | 2 | $4d$ |
| LLaMA / PaLM | SwiGLU | 3 | $8d/3 \approx 2.67d$ |

**Total parameters equivalent** (at $d=4096$): $2 \times 4096 \times 16384 = 2 \times 4096 \times 10923$ — same count.
